# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [21]:
import duckdb
import os

token = os.environ["HF_TOKEN"]

con = duckdb.connect()


con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"SET secret_directory='/tmp'")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{token}'
    )
""")

con.execute("""
    CREATE OR REPLACE VIEW warehouse AS
    SELECT * FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
        hive_partitioning=true
    )
""")

con.execute("SELECT COUNT(*) as total_rows FROM warehouse WHERE month = '2026-03'").fetchdf()

,total_rows
0,9841378


In [7]:
con.execute("DESCRIBE warehouse").fetchdf()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (content_hash_id + client_hash_id) pair observed on one report_date.
Table: fact_content_daily_performance
Time window: 2025-01 through 2026-05 (2026-06 is sealed — never use for development)
Label proxy: will gsc_clicks for this content+client pair increase next month?
Excluded: _sample table — it IS the final month (2026-06), using it = instant leakage

In [8]:
con.execute("""
    SELECT
        MIN(report_date)                    AS earliest_date,
        MAX(report_date)                    AS latest_date,
        COUNT(*)                            AS total_rows,
        COUNT(DISTINCT report_date)         AS distinct_days,
        COUNT(DISTINCT client_hash_id)      AS distinct_clients,
        COUNT(DISTINCT content_hash_id)     AS distinct_pages
    FROM warehouse
    WHERE month = '2026-03'
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,total_rows,distinct_days,distinct_clients,distinct_pages
0,2026-03-01,2026-03-31,9841378,31,55,331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (inputs the model sees):
- gsc_impressions     → how many times the page appeared in search results
- gsc_clicks          → raw clicks from search (observed, fully recorded for past month)
- gsc_avg_position    → average ranking position (lower = better)
- ga4_pageviews       → total page views from Analytics
- sessions_organic    → organic search sessions (engagement signal)

LABEL (what we predict):
- next_month_gsc_clicks → gsc_clicks for the same content+client in the following month
  Proxy: did this page earn more clicks next month? (derived, not a raw column)

CONTEXT (row identity — not model inputs):
- report_date         → date of observation
- month               → partition key, used for filtering only
- client_hash_id      → which client this row belongs to
- content_hash_id     → which page this row describes

EXCLUDED:
- sessions_ai and all ai_* columns → only 5,534 of 9,841,378 rows have values (0.05% fill rate)
- sessions_paid       → only 13,435 rows non-zero (0.14% fill rate), not organic behaviour
- _sample table       → the final month (2026-06), sealed for testing only

In [22]:

con.execute("""
    SELECT
        COUNT(*)                                                        AS total_rows,
        SUM(CASE WHEN sessions_ai > 0 IS TRUE THEN 1 ELSE 0 END)      AS rows_with_ai,
        SUM(CASE WHEN sessions_paid > 0 IS TRUE THEN 1 ELSE 0 END)    AS rows_with_paid,
        SUM(CASE WHEN gsc_clicks > 0 IS TRUE THEN 1 ELSE 0 END)       AS rows_with_clicks,
        SUM(CASE WHEN ga4_pageviews > 0 IS TRUE THEN 1 ELSE 0 END)    AS rows_with_pageviews
    FROM warehouse
    WHERE month = '2026-03'
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ai,rows_with_paid,rows_with_clicks,rows_with_pageviews
0,9841378,5534.0,13435.0,417981.0,413317.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

VERIFICATION QUERIES — every contract claim backed by a query.

Query 1 — Grain check:
Result: No entries (0 duplicate rows).
Proves: one row = one content_hash_id + client_hash_id + report_date. Grain is clean.

Query 2 — Row count and date span:
Result: 9,841,378 rows | 331,437 distinct pages | 55 clients | 2026-03-01 to 2026-03-31
Proves: our slice covers a full calendar month with expected volume.

Query 3 — Availability filter using IS TRUE:
Result: 3,611,061 rows have GSC data | 413,966 have GA4 data | 364,347 have both
Proves: only 37% of rows have GSC data, only 4% have GA4 data.
Most rows are sparse — any model must handle NULLs carefully.
Features should be built only on gsc_data_available IS TRUE rows to avoid noise.

In [23]:

con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM warehouse
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [24]:

con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)                AS start_date,
        MAX(report_date)                AS end_date
    FROM warehouse
    WHERE month = '2026-03'
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_pages,distinct_clients,start_date,end_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [25]:

con.execute("""
    SELECT
        COUNT(*)                                                            AS total_rows,
        SUM(CASE WHEN (gsc_data_available IS TRUE) THEN 1 ELSE 0 END)     AS gsc_available,
        SUM(CASE WHEN (ga4_data_available IS TRUE) THEN 1 ELSE 0 END)     AS ga4_available,
        SUM(CASE WHEN (gsc_data_available IS TRUE)
                  AND (ga4_data_available IS TRUE) THEN 1 ELSE 0 END)     AS both_available
    FROM warehouse
    WHERE month = '2026-03'
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available,both_available
0,9841378,3611061.0,413966.0,364347.0


FIVE FEATURES — availability at decision time:

1. total_impressions  → sum of gsc_impressions for the month. Knowable at decision time
                        because GSC records impressions at end of each reporting day.

2. total_clicks       → sum of gsc_clicks for the month. Knowable at decision time
                        because clicks are fully observed before next month starts.

3. avg_position       → average ranking position across the month. Knowable at decision
                        time because GSC reports position daily, averaged over closed month.

4. ctr                → clicks / impressions for the month. Knowable at decision time
                        because it is derived from two fully observed columns above.

5. active_gsc_days    → count of days the page had GSC data available. Knowable at
                        decision time because it counts past days, not future ones.

All five features are filtered on gsc_data_available IS TRUE — 3,611,061 rows survive.
Feature frame shape: (10, 7) shown here as a sample; full frame has 331,437 pages.

In [26]:

feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        -- Feature 1
        SUM(gsc_impressions)                                                AS total_impressions,
        -- Feature 2
        SUM(gsc_clicks)                                                     AS total_clicks,
        -- Feature 3
        ROUND(AVG(gsc_avg_position), 2)                                     AS avg_position,
        -- Feature 4
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr,
        -- Feature 5
        COUNT(DISTINCT CASE WHEN gsc_data_available IS TRUE
              THEN report_date END)                                          AS active_gsc_days
    FROM warehouse
    WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    LIMIT 10
""").fetchdf()

print(feature_frame.shape)
feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(10, 7)


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,ctr,active_gsc_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.39,0.0018,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.71,0.0000,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.48,0.0000,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.32,0.0042,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.46,0.0058,31
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,14.75,0.0000,21
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,6.34,0.0067,30
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,12.79,0.0000,26
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,4.95,0.0038,31
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,50.39,0.0000,30


LEAKAGE TRAP — lesson performed on real warehouse data:

Step 1: Added leaky_ctr (clicks/impressions from same period as label).
        Correlation with total_clicks = 0.01 — low but still wrong in principle.

Step 2: Added leaky_clicks_feature (same period clicks as a predictor).
        Correlation with total_clicks = 1.0 — perfectly predicts because it IS the label.
        Score jumped to perfect. That is the trap.

Step 3: Deleted both leaky columns. Honest frame has 176,738 rows, 7 clean features.

Rule: if a feature could not be known BEFORE the prediction period starts, it is leakage.
      Same-period clicks cannot predict next-month clicks — they are the same thing.
      Always ask: "could I have known this at the moment I make the decision?"

In [28]:

leaky_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                                 AS total_impressions,
        SUM(gsc_clicks)                                                      AS total_clicks,
        ROUND(AVG(gsc_avg_position), 2)                                      AS avg_position,
        -- THIS IS LEAKAGE: same-period clicks used as a feature
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)            AS leaky_ctr
    FROM warehouse
    WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()


corr = leaky_frame[['total_clicks', 'leaky_ctr']].corr()
print("Correlation with leaky feature:")
print(corr)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Correlation with leaky feature:
              total_clicks  leaky_ctr
total_clicks      1.000000   0.012573
leaky_ctr         0.012573   1.000000


In [29]:

leaky_frame2 = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)                AS total_clicks,
        -- OBVIOUS LEAKAGE: clicks from same period used as "predictor"
        SUM(gsc_clicks) * 1.0         AS leaky_clicks_feature
    FROM warehouse
    WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

corr2 = leaky_frame2[['total_clicks', 'leaky_clicks_feature']].corr()
print("Correlation with obvious leaky feature:")
print(corr2)
print()
print("Score is perfect (1.0) — because the feature IS the label. That is leakage.")

Correlation with obvious leaky feature:
                      total_clicks  leaky_clicks_feature
total_clicks                   1.0                   1.0
leaky_clicks_feature           1.0                   1.0

Score is perfect (1.0) — because the feature IS the label. That is leakage.


In [32]:

honest_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                                AS total_impressions,
        SUM(gsc_clicks)                                                     AS total_clicks,
        ROUND(AVG(gsc_avg_position), 2)                                     AS avg_position,
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr,
        COUNT(DISTINCT CASE WHEN gsc_data_available IS TRUE
              THEN report_date END)                                          AS active_gsc_days
    FROM warehouse
    WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

print("Honest feature frame shape:", honest_frame.shape)
honest_frame.head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,ctr,active_gsc_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.39,0.0018,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.71,0.0000,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.48,0.0000,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.32,0.0042,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.46,0.0058,31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

DATA LIMITS — what this data cannot tell us:

1. Unbalanced history (proven above):
   - 2025-01: 1,297 rows, zero GA4 data — GSC-only early months
   - 2025-06: 329,201 rows, still zero GA4 data
   - 2026-01: GA4 finally appears but only 115,786 of 7,890,817 rows (1.5%)
   - 2026-03: GA4 grows to 413,966 of 9,841,378 rows (4.2%)
   Any model trained across the full window will see very different feature
   availability in early months vs late months — history is not consistent.

2. GSC blind spot:
   GSC only records pages that appeared in search results. Pages with zero
   impressions are invisible — the data is biased toward existing visibility,
   not undiscovered opportunity.

3. Position is directional, not exact:
   gsc_avg_position is averaged across users, devices, and locations.
   It is a directional signal only, not a precise ranking number.

4. Window overlap risk:
   Rolling features built across adjacent months can bleed future signal
   into past observations if partition boundaries are not handled carefully.

In [31]:

con.execute("""
    SELECT
        month,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)  AS gsc_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)  AS ga4_available,
        COUNT(*)                                                       AS total_rows
    FROM warehouse
    WHERE month IN ('2025-01', '2025-06', '2026-01', '2026-03')
    GROUP BY month
    ORDER BY month
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,gsc_available,ga4_available,total_rows
0,2025-01,1297.0,0.0,1297
1,2025-06,329201.0,0.0,329201
2,2026-01,2397143.0,115786.0,7890817
3,2026-03,3611061.0,413966.0,9841378


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.